# Lecture 19: Fourth and final lab, make your helicopter fly
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/engr1050-fall2026/blob/main/NewMaterial/Lecture19/lec19.ipynb)


## Overview

For todays lecture, you will be working in groups of 2-3 and working in Thonny to push code onto the Picos again. Remember that **this Jupyter notebook is not meant to be run in Colab.**

This is our final lab where we will make the helicopter hover. This will be much more about wiring than programming. We have several moving pieces:
- A potentiometer, whose job is to take an angle reading
- A motor, whose voltage supplied will govern the lift from the blades
- A *transistor*, which we will use to throttle the amount of power to the motor as a function of the potentiometer reading
- A battery, to provide power to the motor

In order to do this, we will use the *breadboard* to build up a control circuit. Whereas before we powered lights and buttons off of the pico, the motor requires more power. We will learn how to use the pico to switch the power flowing to the motor through the transistor without having that power flow through the pico itself (it would cook the pico!).

We'll do a couple warmup problems first to help you understand what the breadboard and transistor are and how to use them, and then we'll make our helicopter fly! 

**⚠️⚠️⚠️ Safety⚠️⚠️⚠️:** The motors move pretty quick - there's a good chance you might catch one on your hand. They don't spin fast enough to break skin but it is annoying, so please take care. If you have long hair, tie it back so that it doesn't get wrapped in the motor.

# Using the breadboard

The breadboard lets you easily put circuits together without the need to solder permanent circuits together. Think of it as a way to reconfigurably wire circuits together like legos.

<div align="center">
    <img src="../_shared/Images/breadboard.jpg" alt="Lab Setup" width="600">
</div>

The top-down view on the left shows how you interact with the breadboard. You insert the male end of a jump cable to any one of the holes. The bottom-up view on the right shows how the holes are wired together. Each group of 5 holes (e.g. 21a-e or 30f-j) are connected. Also the two positive and negative bars along the sides are wired.

## Exercise 1. Wire your own LED

So far we used the built in LEDs on the board. Now we will wire our own to practice using the breadboard. An LED (light emitting diode) lights up when current flows from its cathode (short leg) to its anode (long leg).
<div align="center">
    <img src="../_shared/Images/lec19_led.png" alt="Lab Setup" width="400">
</div>

- Use a female-to-male jumper to connect the `GP16` pin to `J1`.
- Use a female-to-male jumper to connect a `GND` pin to `J2`
- Insert the LED into the bread board so that the legs straddle the same two groups of pins (e.g. long leg into `G1`, short leg into `G2`). If it doesn't work you most likely have the LED flipped backwards.
- Run the code below to generate a smoothly pulsating LED.

In [ ]:
from machine import Pin, PWM
import time
import math

led_pwm = PWM(Pin(16))
led_pwm.freq(1000) # Set frequency to 1kHz to modulate brightness


# Smooth sine wave breathing effect
while True:

    for i in range(360):
        # Create sine wave from 0 to 1
        brightness = (math.sin(math.radians(i)) + 1) / 2
        # Convert to 16-bit PWM value
        duty = int(0.5*brightness * 65535)
        led_pwm.duty_u16(duty)
        time.sleep(0.02)  # 20ms for smooth effect

This code uses *pulse width modulation* to make it seem like the LED is smoothly turning off and on. LEDs can only be off or on - we make it seem dim rapidly cycling the LED off and over a 1kHz window. To dim it by half, we leave it on 0.5/1000 s and then off 0.5/1000 s. To dim it 25\%, we leave it on 0.25/1000 s and then off 0.75/1000 s. `duty` is a number between 0 and 65535 that controls the strength of the signal to the LED. 

In the same way, we will throttle power to the motor to control its speed.

**TODO:** Capture a video of the modulated LED.

# Introducing transistors

We will use a transistor to switch the LED/fan off or on. The transistor has three pins. If it is lying on the ground with the model number facing you, the pins are:
- *Left:* `Base`. You connect this to a `GP` pin, and it will open/close the circuit depending on whether the `GP` is active or not.
- *Center:* `Collector`. You will connect this to the power source.
- *Right:* `Emitter`. This connects to the downstream circuit that the collector powers.

Think of the base pin like a little guy with the job to flip the power on a power plant off or on. There could be a huge amount of power flowing through the circuit, so we we need to protect the little guy from getting zapped when he turns the power off/on; that's exactly what a transistor does. It lets power flow from the collector to the emitter with an off/on switch depending on the signal to the base pin.

**Warning:** Be careful with the orientation. My analogy of the little guy that could be zapped is real. The battery pack that we'll use to make the motor fly is enough to fry the Pico and break it. Make sure that your Pico only ever has power flowing between a `GP` pin and the `Base` to keep it out of the hot circuit.


<div align="center">
    <img src="../_shared/Images/lec19_transistor.png" alt="Transistor" width="400">
</div>

## Exercise 2. Throttle power using a transistor

We'll set up a simple circuit now that uses a transistor to control the supply of power to an LED to make it blink.

First, we'll supply power to the breadboard. 
- Use a male-to-female jumper to connect `3V3` to the positive cross bar at the top of the breadboard.
- Use a male-to-female jumper to connect `GND` to the negative cross bar at the bottom of the breadboard.
<div align="center">
    <img src="../_shared/Images/lec_19_ledtransistor_step1.jpg" alt="Transistor step 1" width="400">
</div>
You now have power in the breadboard. Any circuit connected to the positive or negative bar will have power.

Next, set up the transistor.
- With the letters of the transistor facing *toward you*, plug the transistor into `j1-j3`. 
- For the `Base`, connect `GP16` to `g1` - this is where you flip the switch.
- For the `Collector`, connect `g2` to the positive crossbar.
- For the `Emitter`, connect `g3` to `e10` - this is the circuit that you will power.
<div align="center">
    <img src="../_shared/Images/lec_19_ledtransistor_step2.jpg" alt="Transistor step 1" width="400">
</div>

Finally, we'll build a simple circuit toggled by the transistor consisting of an LED connected to ground.
- Insert an LED with the long leg into `c10` and the short leg into `c11`.
- Ground the circuit by connecting `a11` to the negative cross bar.
<div align="center">
    <img src="../_shared/Images/lec_19_ledtransistor_step3.jpg" alt="Transistor step 1" width="400">
</div>


If you've wired everything correctly, our usual blinky LED code below should flash the LED. Unlike the first exercise however, the power to the LED doesn't run through the `GP` pin - it comes off of the power crossbar.

In [ ]:
from machine import Pin
import time

# Set up the LED on GP16 as an output
led = Pin(16, Pin.OUT)

# Blink the LED continuously
while True:
    led.on()          # Turn LED on
    time.sleep(0.5)   # Wait 0.5 seconds
    led.off()         # Turn LED off
    time.sleep(0.5)   # Wait 0.5 seconds

**TODO:** Capture a video of the blinking LED.

# Exercise 3: Fly helicopter fly

OK now that we know what our new pieces do, we're going to build our helicopter. Clear all of your wires, we're going to start over from scratch.

**Step 1. Set up power supply**

1. Connect a battery pack to a pair of alligator clips and male-to-male jumper cable.
2. Connect the positive terminal of the battery (*red line*) to the positive (+) crossbar.
3. Connect the negative terminal of the battery (*black line*) to the negative (-) crossbar.
4. Connect the `GND` pin of the Pico board to the negative crossbar. This makes sure both the breadboard and the Pico have a common ground.

<div align="center">
    <img src="../_shared/Images/lec19_heli1.png" alt="helicopter step 1" width="400">
</div>

**Step 2. Prepare wiring for the motor.**
1. Extend the red (positive) and blue (negative) wires coming off of that DC motor attached to the fan with an alligator clip attached to a male-to-male jumper cable.
2. Be careful - if you get these reversed the motor will blow down instead of up.

<div align="center">
    <img src="../_shared/Images/lec19_heli2.png" alt="helicopter step 2" width="400">
</div>

**Step 3. Wire the transistor.**

1. Insert the transistor into the `j1-3` slots of the breadboard, oriented so that the `TIP31CG` letters are facing you.
2. Connect the Base (left prong) to `GP15`.
3. Connect the Collector (center prong) to the negative lead (blue) of the motor.
4. Connect the Emitter (right prong) to the negative crossbar of the breadboard.
5. Connect the positive lead of the motor (red) to the positive crossbar of the breadboard.

**Careful.** Make sure that you don't accidentally short circuit your Pico/laptop. The alligator clips will transmit power if they accidentally come into contact with each other.

<div align="center">
    <img src="../_shared/Images/lec19_heli3.png" alt="helicopter step 3" width="800">
</div>

**Step 4. Wire the potentiometer.**

Finally, you can wire the potentiometer the same as we always do.
1. Center pin to `GP26`.
2. Outside pins to `3V3` and ground.
3. **Do not connect the potentiometer to the positive cross bar or you will cook your Pico.**

**Final step.**
Run the following code in Thonny and take a video of your helicopter hovering.

In [ ]:
from machine import Pin, PWM, ADC
from time import sleep

pwm = PWM(Pin(15))
pwm.freq(10000)
adc = ADC(Pin(26))

setpoint = 45
invert_flag = True

if invert_flag:
    theta0 = 34000
    theta90 = 9200
else:
    theta0 = 9200
    theta90 = 34000

while True:
    pot_value = adc.read_u16()
    
    theta_value = 90.0*(pot_value-theta0)/(theta90-theta0)

    error = setpoint - theta_value
    control_signal = 65535 * (-1*error / 90)

    out_val = min(65535, int(control_signal))
    out_val = max(0, out_val)
    pwm.duty_u16(out_val)

    
    print(f"theta: {theta_value}, setpoint: {setpoint}, pot: {pot_value}")
    print(f"control {control_signal} Error: {error}, Duty: {out_val}")
    sleep(0.05)

# Submit today's work #
You should have three videos:
1. Your pulsating LED.
2. Your blinking LED modulated by the transistor.
3. Your floating helicopter.

Submit to Canvas using the following [link](https://canvas.upenn.edu/courses/1881448/assignments/14074145), denoting in the text box the names of the folks in your group.